In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [19]:
!wget https://raw.githubusercontent.com/yoosufaamir/HPC-/main/images/Butterfly.png

--2026-10-08 17:07:50--  https://raw.githubusercontent.com/yoosufaamir/HPC-/main/images/Butterfly.png
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30276 (30K) [image/png]
Saving to: ‘Butterfly.png.1’

Butterfly.png.1     100%[===================>]  29.57K  --.-KB/s    in 0.002s  

2026-10-08 17:07:50 (12.9 MB/s) - ‘Butterfly.png.1’ saved [30276/30276]



In [20]:
!wget https://raw.githubusercontent.com/lvandeve/lodepng/master/lodepng.h
!wget https://raw.githubusercontent.com/lvandeve/lodepng/master/lodepng.cpp

--2026-10-08 17:07:53--  https://raw.githubusercontent.com/lvandeve/lodepng/master/lodepng.h
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 108843 (106K) [text/plain]
Saving to: ‘lodepng.h.1’

lodepng.h.1         100%[===================>] 106.29K  --.-KB/s    in 0.01s   

2026-10-08 17:07:53 (10.5 MB/s) - ‘lodepng.h.1’ saved [108843/108843]

--2026-10-08 17:07:53--  https://raw.githubusercontent.com/lvandeve/lodepng/master/lodepng.cpp
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 312689 (305K) [text/plain]
Saving to: ‘lodepng.

In [31]:
!nvcc -o SobelEdge SobelEdge.cu lodepng.cpp

In [32]:
%%writefile SobelEdge.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <dirent.h>
#include <sys/stat.h>
#include <math.h>

#include "lodepng.h"

const int ERROR_EXIT_VALUE = -1;

__global__ void sobelEdge(
    unsigned char *inImg,
    unsigned char *outGx,
    unsigned char *outGy,
    unsigned char *outSobel,
    unsigned int width,
    unsigned int height
)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height)
        return;

    int index = (y * width + x) * 4;

    int GxKernel[3][3] =
    {
        {-1,  0,  1},
        {-2,  0,  2},
        {-1,  0,  1}
    };

    int GyKernel[3][3] =
    {
        {-1, -2, -1},
        { 0,  0,  0},
        { 1,  2,  1}
    };

    float Gx = 0.0f;
    float Gy = 0.0f;

    for (int ky = -1; ky <= 1; ky++)
    {
        for (int kx = -1; kx <= 1; kx++)
        {
            int pixelX = x + kx;
            int pixelY = y + ky;
            float gray = 0.0f;

            if (pixelX >= 0 && pixelX < width && pixelY >= 0 && pixelY < height)
            {
                int pixelIndex = (pixelY * width + pixelX) * 4;
                unsigned char r = inImg[pixelIndex];
                unsigned char g = inImg[pixelIndex + 1];
                unsigned char b = inImg[pixelIndex + 2];
                gray = 0.299f * r + 0.587f * g + 0.114f * b;
            }

            Gx += gray * GxKernel[ky + 1][kx + 1];
            Gy += gray * GyKernel[ky + 1][kx + 1];
        }
    }

    float gxDisplay = Gx + 128.0f;
    if (gxDisplay < 0.0f) gxDisplay = 0.0f;
    if (gxDisplay > 255.0f) gxDisplay = 255.0f;
    unsigned char gxValue = (unsigned char)gxDisplay;

    outGx[index]     = gxValue;
    outGx[index + 1] = gxValue;
    outGx[index + 2] = gxValue;
    outGx[index + 3] = 255;

    float gyDisplay = Gy + 128.0f;
    if (gyDisplay < 0.0f) gyDisplay = 0.0f;
    if (gyDisplay > 255.0f) gyDisplay = 255.0f;
    unsigned char gyValue = (unsigned char)gyDisplay;

    outGy[index]     = gyValue;
    outGy[index + 1] = gyValue;
    outGy[index + 2] = gyValue;
    outGy[index + 3] = 255;

    float magnitude = sqrtf(Gx * Gx + Gy * Gy);
    if (magnitude > 255.0f) magnitude = 255.0f;
    if (magnitude < 0.0f) magnitude = 0.0f;
    unsigned char edge = (unsigned char)magnitude;

    outSobel[index]     = edge;
    outSobel[index + 1] = edge;
    outSobel[index + 2] = edge;
    outSobel[index + 3] = 255;
}

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = (call); \
        if (err != cudaSuccess) { \
            fprintf(stderr, "CUDA error at %s:%d - %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

int hasPngExtension(const char *name)
{
    size_t len = strlen(name);
    if (len < 4) return 0;
    const char *ext = name + len - 4;
    return (ext[0] == '.' && (ext[1] == 'p' || ext[1] == 'P') && (ext[2] == 'n' || ext[2] == 'N') && (ext[3] == 'g' || ext[3] == 'G'));
}

int processImage(const char *inPath, const char *outGxPath, const char *outGyPath, const char *outSobelPath)
{
    unsigned char *c_inImg = NULL;
    unsigned char *c_gxImg = NULL;
    unsigned char *c_gyImg = NULL;
    unsigned char *c_sobelImg = NULL;
    unsigned int width = 0, height = 0;

    unsigned int error = lodepng_decode32_file(&c_inImg, &width, &height, inPath);
    if (error) {
        printf("Error decoding %s: %u %s\n", inPath, error, lodepng_error_text(error));
        return ERROR_EXIT_VALUE;
    }

    printf("Processing: %s (%u x %u)\n", inPath, width, height);
    unsigned int imageSize = 4 * width * height;

    c_gxImg = (unsigned char *)malloc(imageSize);
    c_gyImg = (unsigned char *)malloc(imageSize);
    c_sobelImg = (unsigned char *)malloc(imageSize);

    if (c_gxImg == NULL || c_gyImg == NULL || c_sobelImg == NULL) {
        printf("CPU memory allocation failed.\n");
        free(c_inImg); free(c_gxImg); free(c_gyImg); free(c_sobelImg);
        return ERROR_EXIT_VALUE;
    }

    unsigned char *d_inImg, *d_gxImg, *d_gyImg, *d_sobelImg;
    CUDA_CHECK(cudaMalloc((void **)&d_inImg, imageSize));
    CUDA_CHECK(cudaMalloc((void **)&d_gxImg, imageSize));
    CUDA_CHECK(cudaMalloc((void **)&d_gyImg, imageSize));
    CUDA_CHECK(cudaMalloc((void **)&d_sobelImg, imageSize));

    CUDA_CHECK(cudaMemcpy(d_inImg, c_inImg, imageSize, cudaMemcpyHostToDevice));

    dim3 blockSize(16, 16);
    dim3 gridSize((width + blockSize.x - 1) / blockSize.x, (height + blockSize.y - 1) / blockSize.y);

    sobelEdge<<<gridSize, blockSize>>>(d_inImg, d_gxImg, d_gyImg, d_sobelImg, width, height);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(c_gxImg, d_gxImg, imageSize, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(c_gyImg, d_gyImg, imageSize, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(c_sobelImg, d_sobelImg, imageSize, cudaMemcpyDeviceToHost));

    lodepng_encode32_file(outGxPath, c_gxImg, width, height);
    lodepng_encode32_file(outGyPath, c_gyImg, width, height);
    lodepng_encode32_file(outSobelPath, c_sobelImg, width, height);

    CUDA_CHECK(cudaFree(d_inImg));
    CUDA_CHECK(cudaFree(d_gxImg));
    CUDA_CHECK(cudaFree(d_gyImg));
    CUDA_CHECK(cudaFree(d_sobelImg));

    free(c_inImg); free(c_gxImg); free(c_gyImg); free(c_sobelImg);
    return 0;
}

int main(int argc, char *argv[])
{
    const char *inputFolder = "images";
    const char *outputFolder = "output";
    if (argc > 2) {
        inputFolder = argv[1];
        outputFolder = argv[2];
    }
    mkdir(outputFolder, 0777);
    DIR *dir = opendir(inputFolder);
    if (dir == NULL) {
        printf("Could not open input folder: %s\n", inputFolder);
        return ERROR_EXIT_VALUE;
    }
    int processed = 0, failed = 0;
    struct dirent *entry;
    while ((entry = readdir(dir)) != NULL) {
        if (!hasPngExtension(entry->d_name)) continue;
        char inPath[600], outGxPath[600], outGyPath[600], outSobelPath[600];
        snprintf(inPath, sizeof(inPath), "%s/%s", inputFolder, entry->d_name);
        char baseName[500];
        size_t nameLength = strlen(entry->d_name);
        strncpy(baseName, entry->d_name, nameLength - 4);
        baseName[nameLength - 4] = '\0';

        snprintf(outGxPath, sizeof(outGxPath), "%s/%s_gradient_x.png", outputFolder, baseName);
        snprintf(outGyPath, sizeof(outGyPath), "%s/%s_gradient_y.png", outputFolder, baseName);
        snprintf(outSobelPath, sizeof(outSobelPath), "%s/%s_sobel.png", outputFolder, baseName);

        if (processImage(inPath, outGxPath, outGyPath, outSobelPath) == 0) {
            processed++;
        } else {
            failed++;
        }
    }
    closedir(dir);
    printf("\nDone. %d image(s) processed successfully, %d failed.\n", processed, failed);
    return 0;
}

Overwriting SobelEdge.cu


In [34]:
%%writefile SobelImage.py
import os
import shutil
import subprocess
import matplotlib.pyplot as plt
import cv2

# Ensure clean, fresh folders each run
if os.path.exists("images"):
    shutil.rmtree("images")
if os.path.exists("output"):
    shutil.rmtree("output")

os.makedirs("images", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Configure to process and display ONLY Butterfly.png
image_files = ["Butterfly.png"]
for img in image_files:
    if os.path.exists(img):
        shutil.copy(img, os.path.join("images", img))

print("Compiling CUDA program...")
subprocess.run(["nvcc", "SobelEdge.cu", "lodepng.cpp", "-o", "SobelEdge"], check=True)

print("\nRunning CUDA Sobel Edge Detection on Butterfly image...")
subprocess.run(["./SobelEdge", "images", "output"], check=True)

for img_file in image_files:
    base_name = os.path.splitext(img_file)[0]
    original_path = f"images/{img_file}"
    gx_path = f"output/{base_name}_gradient_x.png"
    gy_path = f"output/{base_name}_gradient_y.png"
    sobel_path = f"output/{base_name}_sobel.png"

    if not all([os.path.exists(original_path), os.path.exists(gx_path), os.path.exists(gy_path), os.path.exists(sobel_path)]):
        print(f"Missing output files for {base_name}")
        continue

    original = cv2.cvtColor(cv2.imread(original_path), cv2.COLOR_BGR2RGB)
    gradient_x = cv2.cvtColor(cv2.imread(gx_path), cv2.COLOR_BGR2RGB)
    gradient_y = cv2.cvtColor(cv2.imread(gy_path), cv2.COLOR_BGR2RGB)
    sobel = cv2.cvtColor(cv2.imread(sobel_path), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].imshow(original)
    axes[0, 0].set_title("Original Image", fontsize=13)
    axes[0, 0].axis("off")

    axes[0, 1].imshow(gradient_x)
    axes[0, 1].set_title("Gradient in X direction", fontsize=13)
    axes[0, 1].axis("off")

    axes[1, 0].imshow(gradient_y)
    axes[1, 0].set_title("Gradient in Y direction", fontsize=13)
    axes[1, 0].axis("off")

    axes[1, 1].imshow(sobel)
    axes[1, 1].set_title("Sobel Edge Detection", fontsize=13)
    axes[1, 1].axis("off")

    fig.suptitle(f"Sobel Edge Detection - {base_name}", fontsize=16)
    plt.tight_layout()
    plt.show()

Writing SobelImage.py


In [35]:
!python3 SobelImage.py

Compiling CUDA program...

Running CUDA Sobel Edge Detection on Butterfly image...
Processing: images/Butterfly.png (348 x 393)

Done. 1 image(s) processed successfully, 0 failed.
Figure(1200x800)
